In [31]:
import pandas as pd
import os

In [32]:
root = "/Volumes/ArxivE/_Tobias/LSM980/Mitotic_Stopwatch" + "/"
outputFolder = root + "DataFrames" + "/"

df = pd.read_csv('/Volumes/ArxivE/_Tobias/LSM980/Mitotic_Stopwatch/DataFrames/MainDataFrame_Lineages.csv')

In [33]:
def getonsets(filepath, position, dataset):
    onsets = pd.read_csv(filepath)
    onsets["Source_ID"] = onsets.Label.str.split(":").str.get(1).astype(int)
    onsets["Position"] = position
    onsets["dataset"] = dataset
    onsets.rename(columns = {"Slice": "Frame_onset"}, inplace = True)
    return onsets

# Define metadata for each file
onsets_info = [
    {"date": "20250627", "position": 3, "file": "20250627_IM_H2B-GFP_MitoStop_02_onsets.csv"},
    {"date": "20250627", "position": 4, "file": "20250627_IM_H2B-GFP_MitoStop_02_onsets.csv"},
    {"date": "20250704", "position": 1, "file": "20250704_IM_H2B-GFP_MitoStop_03_onsets.csv"},
    {"date": "20250704", "position": 2, "file": "20250704_IM_H2B-GFP_MitoStop_03_onsets.csv"},
    {"date": "20250704", "position": 3, "file": "20250704_IM_H2B-GFP_MitoStop_03_onsets.csv"},
    {"date": "20250704", "position": 4, "file": "20250704_IM_H2B-GFP_MitoStop_03_onsets.csv"},
    {"date": "20250704", "position": 5, "file": "20250704_IM_H2B-GFP_MitoStop_03_onsets.csv"},
    {"date": "20250704", "position": 6, "file": "20250704_IM_H2B-GFP_MitoStop_03_onsets.csv"},
    {"date": "20250704", "position": 7, "file": "20250704_IM_H2B-GFP_MitoStop_03_onsets.csv"},
]

# Generate and combine dataframes
df_onsets_list = []

for info in onsets_info:
    folder = f"{root}/{info['date']}/Position_{info['position']}_MonastrolWashout/Concatenated"
    filepath = os.path.join(folder, info["file"])
    df_single = getonsets(filepath, info["position"], info["date"])
    df_onsets_list.append(df_single)

df_onsets = pd.concat(df_onsets_list, ignore_index = True)

In [34]:
df["Frame"] = df.Frame.astype(int) + 1 #otherwise the trackmate and mitotic timing frames are not aligned

df = df.merge(df_onsets, how = "outer")

df.head()
    

,Unnamed: 0,Track_ID,Source_ID,Splitting_event,Lineage,Track_Coordinate_X,Track_Coordinate_Y,Frame,Position,Dataset,...,Cell_Cycle_hours,Seen_sister,Random_ID,Seen_granny,,Label,X,Y,Frame_onset,dataset
0,0,0,2011,True,2011,444.566899,406.863649,66,3,20250627,...,NaN,NaN,2524,NaN,22,tomeasure:2011:11,171.529,747.022,11,20250627
1,0,0,2012,True,2012,761.937434,648.137100,59,3,20250704,...,NaN,NaN,5249,NaN,52,tomeasure:2012:14,819.528,643.027,14,20250704
2,2,0,2046,True,2012.2046,739.287853,889.272274,461,3,20250704,...,46.900000,NaN,4915,NaN,22,tomeasure:2046:444,742.464,919.794,444,20250704
3,4,1,2081,True,2081,580.740785,683.768758,58,3,20250704,...,NaN,NaN,5249,NaN,46,tomeasure:2081:48,568.449,727.549,48,20250704
4,2,0,2105,True,2011.2105,452.646167,440.838007,527,3,20250627,...,53.783333,True,6515,NaN,76,tomeasure:2105:500,454.096,473.569,500,20250627


In [35]:
df["Frames_in_mitosis"] = (df.Frame.astype(int) - df.Frame_onset.astype(int))
df["Mitotic_duration_mins"] = df.Frames_in_mitosis * 7

In [36]:
def getMitoticDurationMother(x, dataframe = df):
    if x.Generation < 2:
        return None
    else:
        mother_row = dataframe.loc[
            (dataframe.dataset == x.dataset) &
            (dataframe.Position == x.Position) &
            (dataframe.Source_ID == x.Mother_ID)
        ]
        
        if len(mother_row) == 1:
            return mother_row.iloc[0].Mitotic_duration_mins
        else:
            return None


def getMetaphaseArrest(x):
    if x > 100:
        return "> 100 min"
   # elif x > 45:
   #     return "045-90"
    else:
        return "< 100 min"

df["Mother_Mitotic_duration_mins"] = df.apply(getMitoticDurationMother, axis = 1)
df["Mother_arrested"] = df.Mother_Mitotic_duration_mins.apply(getMetaphaseArrest)

df.head()

,Unnamed: 0,Track_ID,Source_ID,Splitting_event,Lineage,Track_Coordinate_X,Track_Coordinate_Y,Frame,Position,Dataset,...,,Label,X,Y,Frame_onset,dataset,Frames_in_mitosis,Mitotic_duration_mins,Mother_Mitotic_duration_mins,Mother_arrested
0,0,0,2011,True,2011,444.566899,406.863649,66,3,20250627,...,22,tomeasure:2011:11,171.529,747.022,11,20250627,55,385,NaN,< 100 min
1,0,0,2012,True,2012,761.937434,648.137100,59,3,20250704,...,52,tomeasure:2012:14,819.528,643.027,14,20250704,45,315,NaN,< 100 min
2,2,0,2046,True,2012.2046,739.287853,889.272274,461,3,20250704,...,22,tomeasure:2046:444,742.464,919.794,444,20250704,17,119,315.0,> 100 min
3,4,1,2081,True,2081,580.740785,683.768758,58,3,20250704,...,46,tomeasure:2081:48,568.449,727.549,48,20250704,10,70,NaN,< 100 min
4,2,0,2105,True,2011.2105,452.646167,440.838007,527,3,20250627,...,76,tomeasure:2105:500,454.096,473.569,500,20250627,27,189,385.0,> 100 min


In [37]:
df.to_csv(outputFolder + "MainDataFrame_MitoticStopwatch.csv")

print("Finished.")

Finished.
